# Exploratory Data Analysis on Electricity

## 0. Get Setup

In [7]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

## 1. Download Data

In [8]:
# Download data from google drive with gdown
try:
    import gdown
    print(f"[INFO] gdown version: {gdown.__version__}")
except:
    print("[INFO] Couldn't find gdown, installing it...")
    !pip install -q gdown
    import gdown
    print(f"[INFO] gdown version: {gdown.__version__}")

!mkdir -p /data/datasets/public/electricity/
!gdown 1jinfTAApPyuyvW1P1hUDpI3rl0Jq8in1 -O /data/datasets/public/electricity/

[INFO] gdown version: 5.2.1
Downloading...
From: https://drive.google.com/uc?id=1jinfTAApPyuyvW1P1hUDpI3rl0Jq8in1
To: /data/datasets/public/electricity/electricity.csv
100% 95.6M/95.6M [00:01<00:00, 76.3MB/s]


## 2. ACF/PACF

In [9]:
# Load and parse data
df = pd.read_csv('/data/datasets/public/electricity/electricity.csv')
df['date'] = pd.to_datetime(df['date'])
df.set_index('date', inplace=True)
df.head()

,0,1,2,3,4,5,6,7,8,9,...,311,312,313,314,315,316,317,318,319,OT
date,,,,,,,,,,,,,,,,,,,,,
2016-07-01 02:00:00,14.0,69.0,234.0,415.0,215.0,1056.0,29.0,840.0,226.0,265.0,...,676.0,372.0,80100.0,4719.0,5002.0,48.0,38.0,1558.0,182.0,2162.0
2016-07-01 03:00:00,18.0,92.0,312.0,556.0,292.0,1363.0,29.0,1102.0,271.0,340.0,...,805.0,452.0,95200.0,4643.0,6617.0,65.0,47.0,2177.0,253.0,2835.0
2016-07-01 04:00:00,21.0,96.0,312.0,560.0,272.0,1240.0,29.0,1025.0,270.0,300.0,...,817.0,430.0,96600.0,4285.0,6571.0,64.0,43.0,2193.0,218.0,2764.0
2016-07-01 05:00:00,20.0,92.0,312.0,443.0,213.0,845.0,24.0,833.0,179.0,211.0,...,801.0,291.0,94500.0,4222.0,6365.0,65.0,39.0,1315.0,195.0,2735.0
2016-07-01 06:00:00,22.0,91.0,312.0,346.0,190.0,647.0,16.0,733.0,186.0,179.0,...,807.0,279.0,91300.0,4116.0,6298.0,75.0,40.0,1378.0,191.0,2721.0


In [10]:
import os

# Create directory for plots
os.makedirs('acf_pacf_plots', exist_ok=True)

series_to_plot = [11, 25, 81, 4, 24, 27, 152, 154, 237, 238, 206, 202, 37, 8, 113, 114]

for series_num in series_to_plot:
    series_data = df.iloc[:, series_num]

    # ACF plot
    fig, ax = plt.subplots(1, 1, figsize=(10, 3))
    plot_acf(series_data, lags=200, ax=ax)
    ax.set_title(f'Series {series_num} - ACF')
    plt.tight_layout()
    plt.savefig(f'acf_pacf_plots/series_{series_num}_acf.png', dpi=150)  # Save to folder
    plt.close()

    # PACF plot
    fig, ax = plt.subplots(1, 1, figsize=(10, 3))
    plot_pacf(series_data, lags=200, ax=ax)
    ax.set_title(f'Series {series_num} - PACF')
    plt.tight_layout()
    plt.savefig(f'acf_pacf_plots/series_{series_num}_pacf.png', dpi=150)  # Save to folder
    plt.close()

print("Saved all plots to acf_pacf_plots/")

Saved all plots to acf_pacf_plots/


## 3. Periodogram

In [11]:
from scipy import signal

for series_num in series_to_plot:
    series_data = df.iloc[:, series_num].values

    # Compute periodogram
    freqs, power = signal.periodogram(series_data, fs=1.0)  # fs=1.0 for hourly sampling

    # Convert frequency to period (hours)
    periods = 1 / freqs[1:]  # Skip DC component (freq=0)
    power = power[1:]

    # Plot
    fig, ax = plt.subplots(figsize=(10, 3))
    ax.plot(periods, power)
    ax.set_xlim(0, 250)
    ax.set_xlabel('Period (hours)')
    ax.set_ylabel('Power')
    ax.set_title(f'Series {series_num} - Periodogram')
    ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(f'acf_pacf_plots/series_{series_num}_periodogram.png', dpi=150)
    plt.close()

In [12]:
# Zip the plots folder and download locally
from google.colab import files
!zip -r acf_pacf_plots.zip ./acf_pacf_plots/
files.download('acf_pacf_plots.zip')

  adding: acf_pacf_plots/ (stored 0%)
  adding: acf_pacf_plots/series_25_pacf.png (deflated 17%)
  adding: acf_pacf_plots/series_152_periodogram.png (deflated 21%)
  adding: acf_pacf_plots/series_4_acf.png (deflated 13%)
  adding: acf_pacf_plots/series_154_acf.png (deflated 13%)
  adding: acf_pacf_plots/series_206_pacf.png (deflated 17%)
  adding: acf_pacf_plots/series_113_acf.png (deflated 15%)
  adding: acf_pacf_plots/series_24_pacf.png (deflated 18%)
  adding: acf_pacf_plots/series_202_pacf.png (deflated 17%)
  adding: acf_pacf_plots/series_114_acf.png (deflated 17%)
  adding: acf_pacf_plots/series_11_acf.png (deflated 10%)
  adding: acf_pacf_plots/series_11_pacf.png (deflated 16%)
  adding: acf_pacf_plots/series_4_periodogram.png (deflated 18%)
  adding: acf_pacf_plots/series_37_acf.png (deflated 14%)
  adding: acf_pacf_plots/series_4_pacf.png (deflated 18%)
  adding: acf_pacf_plots/series_8_pacf.png (deflated 17%)
  adding: acf_pacf_plots/series_25_periodogram.png (deflated 11%)
 

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>